In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde
from matplotlib.patches import Patch

# =========================
# Inputs
# =========================
base_path = "/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/MCS_PRECEFF/"
files = {
    "CWP": ("Obs_cwp_PRECEFF_mcs_20160809-20160909_Asia_timeavg.nc",
            "Obs_cwp_PRECEFF_non_mcs_20160809-20160909_Asia_timeavg.nc"),
    "CIW": ("Obs_ciw_PRECEFF_mcs_20160809-20160909_Asia_timeavg.nc",
            "Obs_ciw_PRECEFF_non_mcs_20160809-20160909_Asia_timeavg.nc"),
    "CLW": ("Obs_clw_PRECEFF_mcs_20160809-20160909_Asia_timeavg.nc",
            "Obs_clw_PRECEFF_non_mcs_20160809-20160909_Asia_timeavg.nc")
}

# x-axis ordering and the LaTeX labels you want
condensates = ["CWP", "CIW", "CLW"]
condensate_labels = {
    "CWP": r"$\epsilon_{\mathrm{cwp}}$",
    "CIW": r"$\epsilon_{i}$",
    "CLW": r"$\epsilon_{\ell}$"
}

# =========================
# Load + build dataframe
# =========================
data_records = []
for label, (mcs_file, non_mcs_file) in files.items():
    mcs_vals = xr.open_dataset(base_path + mcs_file)["PRECEFF_TIMEAVG"].values.flatten() * 3600
    non_mcs_vals = xr.open_dataset(base_path + non_mcs_file)["PRECEFF_TIMEAVG"].values.flatten() * 3600

    mcs_vals = mcs_vals[~np.isnan(mcs_vals)]
    non_mcs_vals = non_mcs_vals[~np.isnan(non_mcs_vals)]

    data_records.extend([{"value": v, "type": "MCS", "condensate": label} for v in mcs_vals])
    data_records.extend([{"value": v, "type": "Non-MCS", "condensate": label} for v in non_mcs_vals])

df = pd.DataFrame(data_records)
df.columns = df.columns.str.strip()  # clean column names (safe)

# =========================
# Plot styling
# =========================
sns.set(style="ticks", context="talk")

fig, ax = plt.subplots(figsize=(16, 9))

types = ["MCS", "Non-MCS"]
colors = {"MCS": "cyan", "Non-MCS": "orange"}
width = 0.3
y_offset = 0.5   # choose what looks best (0.3–1.0 usually works)


# =========================
# Plot loop
# =========================
for i, cond in enumerate(condensates):
    for j, system in enumerate(types):
        vals = df[(df["condensate"] == cond) & (df["type"] == system)]["value"].dropna().to_numpy()
        pos = i + (j - 0.5) * width  # center MCS and Non-MCS around each condensate

        if len(vals) < 2:
            continue

        # ---- Smoothed violin (manual) ----
        kde = gaussian_kde(vals, bw_method=0.3)
        #ymin = max(0, vals.min() - 0.3 * vals.std())
        ymin = vals.min() - 0.3 * vals.std()
       # ymax = vals.max() + 0.1 * vals.std()
        ymax = vals.max() + 0.1 * vals.std()
        y = np.linspace(ymin, ymax, 1000)
        v = kde(y)
        v = v / v.max() * width * 0.9

        if system == "MCS":
            ax.fill_betweenx(y, pos, pos - v, facecolor=colors[system], alpha=0.5, edgecolor="none")
        else:
            ax.fill_betweenx(y, pos, pos + v, facecolor=colors[system], alpha=0.5, edgecolor="none")

        # ---- Boxplot summary (manual) ----
        #q1 = np.percentile(vals, 25)
        q1 = np.percentile(vals, 5)
        q2 = np.median(vals)
        #q3 = np.percentile(vals, 75)
        q3 = np.percentile(vals, 95)
        iqr = q3 - q1

        lower = max(vals.min(), q1 - 3.5 * iqr)
        upper = min(vals.max(), q3 + 3.5 * iqr)

        ax.plot([pos - 0.05, pos + 0.05], [q1, q1], color="k")
        ax.plot([pos - 0.05, pos + 0.05], [q3, q3], color="k")
        ax.plot([pos - 0.05, pos - 0.05], [q1, q3], color="k")
        ax.plot([pos + 0.05, pos + 0.05], [q1, q3], color="k")
        ax.plot([pos - 0.05, pos + 0.05], [q2, q2], color="k")  # median

        # whiskers
        ax.plot([pos, pos], [q3, upper], color="k", linewidth=3.5)
        ax.plot([pos, pos], [q1, lower], color="k", linewidth=3.5)

        # scatter points
        ax.scatter(np.full(len(vals), pos), vals, color=colors[system], s=3, alpha=0.2, zorder=1)

# =========================
# Axis formatting
# =========================
ax.set_xticks(range(len(condensates)))
ax.set_xticklabels([condensate_labels[c] for c in condensates], fontsize=40)
ax.tick_params(axis="y", labelsize=40)

ax.set_xlim(-0.5, len(condensates) - 0.5)
ax.set_ylabel(r"$\epsilon$ [hr$^{-1}$]", fontsize=40)
ax.set_xlabel("", fontsize=40)
ax.set_ylim(0, 13)

ax.yaxis.grid(True, linestyle=":", linewidth=0.8, color="gray")
ax.set_axisbelow(True)

# =========================
# Legend
# =========================
legend_elements = [
    Patch(facecolor=colors["MCS"], label="MCS", alpha=0.5),
    Patch(facecolor=colors["Non-MCS"], label="Non-MCS", alpha=0.5),
]
ax.legend(handles=legend_elements, title="", loc="upper left", fontsize=23, title_fontsize=23)

sns.despine()
plt.tight_layout()
#plt.show()

# Uncomment to save:
plt.savefig("/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/MCS_Paper_Plots/FigureS2_MCS_precipitation_violin.png",
             dpi=150, bbox_inches="tight")
plt.savefig("/xdisk/sylvia/temakgoale/MCS_TRACKS_DATA/MCS_Paper_Plots/FigureS2_MCS_precipitation_violin.pdf",
             dpi=50, bbox_inches="tight")
